# Unit 10 - Bandits (Exercise) · **V2 material**

**Atoms:** `U10-A13` · **Runtime:** ~25 seconds

## Without code

Bandit cumulative reward > fixed split; bandit CI width on control arm > fixed CI width.

## 1. The question

Implement epsilon-greedy (`epsilon=0.1`) vs fixed 50/50 and compare reward and CI width.

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

In [ ]:
rates = np.array([0.08, 0.11])
n_pulls = 4000
epsilon = 0.1

## 4. TODO - fixed split

Return `fixed_total` conversions over `n_pulls`.

In [ ]:
fixed_total = None  # TODO
assert fixed_total is not None
print('Fixed total:', fixed_total)
assert 280 < fixed_total < 450

## 5. TODO - epsilon-greedy

Track arm pulls and rewards. Return `eg_total` and `eg_control_n` (pulls on arm 0).

In [ ]:
eg_total = None  # TODO
eg_control_n = None  # TODO
assert eg_total is not None and eg_control_n is not None
print('Epsilon-greedy total:', eg_total, 'control pulls:', eg_control_n)
assert eg_total > fixed_total
assert eg_control_n < n_pulls * 0.35

## 6. TODO - CI width

Compute 95% CI width on arm 0 for both policies. `eg_width` should exceed `fixed_width`.

In [ ]:
fixed_width = None  # TODO
eg_width = None  # TODO
assert fixed_width is not None and eg_width is not None
assert eg_width > fixed_width
print('CI widths - fixed:', round(fixed_width, 4), 'epsilon-greedy:', round(eg_width, 4))

**Takeaway:** Bandits optimise traffic, not inference. **Unit:** [V2 unit 10](../V2/units/unit-10-analysis-decision-and-ethics/README.md)

## Hints

With prob `epsilon`, explore randomly; else pull current best empirical arm.

## Spoiler

```python
# Fixed 50/50 split
fixed_arms = np.random.binomial(1, 0.5, n_pulls)
fixed_rewards = np.random.binomial(1, rates[fixed_arms])
fixed_total = int(fixed_rewards.sum())
fixed_control_n = int((fixed_arms == 0).sum())
fixed_control_p = fixed_rewards[fixed_arms == 0].mean()

# Epsilon-greedy
pulls = np.zeros(2, dtype=int)
successes = np.zeros(2, dtype=int)
warmup = 50                      # try each arm before trusting any average
for i in range(n_pulls):
    if i < 2 * warmup or np.random.random() < epsilon:
        arm = i % 2 if i < 2 * warmup else np.random.randint(2)
    else:
        arm = int(np.argmax(successes / pulls))
    reward = np.random.binomial(1, rates[arm])
    pulls[arm] += 1
    successes[arm] += reward
eg_total = int(successes.sum())
eg_control_n = int(pulls[0])

def ci_width(p, n):
    return 2 * 1.96 * np.sqrt(p * (1 - p) / n)

fixed_width = ci_width(fixed_control_p, fixed_control_n)
eg_width = ci_width(successes[0] / pulls[0], pulls[0])
```